In [ ]:
from tcp_enrichment_tools import load_file
import logging
from algo_tcp import add_hajime_column, add_masscan_column, add_mirai_column, add_unicorn_column, add_zmap_column

In [ ]:
csv_file = "test-00_splitfiles/darknet_20251021_mod_non_fragmented_6.csv.gz"

In [ ]:
# Logger configuration
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.DEBUG)

In [ ]:
df = load_file(csv_file, logger)

In [ ]:
print("CSV DataFrame:")
print(df.head())

In [ ]:
df.columns

### Mirai

In [ ]:
df = add_mirai_column(df)

In [ ]:
# Count the number of rows with Mirai traffic
mirai_count = df['mirai'].sum()
print(f"Number of rows with Mirai traffic: {mirai_count} out of a total of {len(df)} rows")

### ZMAP

In [ ]:
df = add_zmap_column(df)

In [ ]:
zmap_count = df['zmap'].sum()
print(f"Number of rows with ZMap traffic: {zmap_count} out of a total of {len(df)} rows")

### MasScan

In [ ]:
df = add_masscan_column(df)

In [ ]:
masscan_count = df['masscan'].sum()
print(f"Number of rows with Masscan traffic: {masscan_count} out of a total of {len(df)} rows")

### Hajime

In [ ]:
df = add_hajime_column(df)

In [ ]:
# Count Hajime traffic
hajime_count = len(df[df['hajime'] == True])
hajime_possible_count = len(df[(df['hajime_possible']) & (df['hajime'] != True)])
print(f"Number of rows with Hajime traffic: {hajime_count} and there are also {hajime_possible_count} possible ones. Total: {hajime_count + hajime_possible_count}")

### Unicorn

In [ ]:
df = add_unicorn_column(df)
unicorn_count = df['unicorn'].sum()
print(f"Number of rows with Unicorn traffic: {unicorn_count} out of a total of {len(df)} rows")

In [ ]:
# Run classify_unicorn_traffic_2
# df = add_unicorn_column_2(df)
# unicorn2_count = df['unicorn2'].sum()
# print(f"Number of rows with unicorn2 traffic: {unicorn2_count} out of a total of {len(df)} rows")

In [ ]:
# Compare ALL Unicorn detection functions
print("=" * 70)
print("UNICORN DETECTION FUNCTIONS COMPARISON")
print("=" * 70)
print()
print("Rows detected by each function:")
print(f"  unicorn        (original, all pairs, 100 max):  {df['unicorn'].sum():>8} rows")
# print(f"  unicorn2       (new5, 10 comparisons):          {df['unicorn2'].sum():>8} rows")
print()
print("Unique IPs detected by each function:")
print(f"  unicorn:        {df[df['unicorn']]['ip_src'].nunique():>8} IPs")
# print(f"  unicorn2:       {df[df['unicorn2']]['ip_src'].nunique():>8} IPs")
print()
print("-" * 70)
print("Matches between pairs (rows):")
print("-" * 70)
# print(f"  unicorn & unicorn2:       {(df['unicorn'] & df['unicorn2']).sum():>8}")
print()
print("-" * 70)
print("Matches between pairs (unique IPs):")
print("-" * 70)
# print(f"  unicorn & unicorn2:       {df[df['unicorn'] & df['unicorn2']]['ip_src'].nunique():>8} IPs")
print()
print("=" * 70)

In [ ]:
# Analyze (show) unicorn records that are not in unicorn2. I see it's Unicorn and use unicorn as valid.
# x = df[(df['unicorn'] == True) & (df['unicorn2'] == False) & (df['ip_src'] == '195.178.110.15')]
# x.to_csv('unicorn_not_unicorn2_195.178.110.15.csv', index=False, sep=';')

### Analysis

In [ ]:
from itertools import combinations

# Classification columns
classification_cols = ['mirai', 'zmap', 'masscan', 'hajime', 'unicorn']

print("=" * 80)
print("TRAFFIC CLASSIFICATION SUMMARY TABLE")
print("=" * 80)
print(f"\nTotal records in DataFrame: {len(df):,}")
print()

# 1. Count by each individual column
print("-" * 80)
print("RECORDS BY TRAFFIC TYPE (individual)")
print("-" * 80)
for col in classification_cols:
    count = df[col].sum()
    pct = (count / len(df)) * 100
    print(f"  {col:12}: {count:>10,} records ({pct:>6.2f}%)")

# 2. Records without any classification
print()
print("-" * 80)
print("UNCLASSIFIED RECORDS")
print("-" * 80)
no_classification = ~df[classification_cols].any(axis=1)
no_class_count = no_classification.sum()
no_class_pct = (no_class_count / len(df)) * 100
print(f"  Without any classification: {no_class_count:>10,} records ({no_class_pct:>6.2f}%)")

# 3. Record count by number of classifications
print()
print("-" * 80)
print("RECORDS BY NUMBER OF SIMULTANEOUS CLASSIFICATIONS")
print("-" * 80)
df['_num_classifications'] = df[classification_cols].sum(axis=1)
for n in range(7):
    count = (df['_num_classifications'] == n).sum()
    pct = (count / len(df)) * 100
    print(f"  {n} classifications: {count:>10,} records ({pct:>6.2f}%)")

# 4. All multiple classification combinations
print()
print("-" * 80)
print("MULTIPLE CLASSIFICATION COMBINATIONS")
print("-" * 80)

# For each combination size (from 6 to 2)
for size in range(6, 1, -1):
    print(f"\n  === Combinations of {size} classifications ===")
    combos = list(combinations(classification_cols, size))
    found_any = False

    for combo in combos:
        # Records that have EXACTLY these classifications (all True, the rest False)
        mask_exact = df[list(combo)].all(axis=1)
        other_cols = [c for c in classification_cols if c not in combo]
        if other_cols:
            mask_exact = mask_exact & ~df[other_cols].any(axis=1)

        # Records that have AT LEAST these classifications (all True)
        mask_at_least = df[list(combo)].all(axis=1)

        count_exact = mask_exact.sum()
        count_at_least = mask_at_least.sum()

        if count_at_least > 0:
            found_any = True
            combo_str = " + ".join(combo)
            print(f"    {combo_str}:")
            print(f"      - Exactly these {size}: {count_exact:>8,}")
            print(f"      - At least these {size}:    {count_at_least:>8,}")

    if not found_any:
        print("    (no combination found)")

# Clean up temporary column
del df['_num_classifications']

print()
print("=" * 80)

In [ ]:
# This cell shows Mirai and Hajime packets, only the tcp_options field, grouped and counted.
mirai_hajime_df = df[(df['mirai'] == True) | (df['hajime'] == True)]
mirai_hajime_grouped = mirai_hajime_df.groupby('tcp_options').size().reset_index(name='count').sort_values(by='count', ascending=False)
print("Unique TCP options in Mirai and Hajime traffic:")
print(mirai_hajime_grouped)

In [ ]:
# This cell shows ZMap packets, only the tcp_options field, grouped and counted.
zmap_df = df[df['zmap'] == True]
zmap_grouped = zmap_df.groupby('tcp_options').size().reset_index(name='count').sort_values(by='count', ascending=False)
print("Unique TCP options in ZMap traffic:")
print(zmap_grouped)

In [ ]:
# This cell shows Masscan packets, only the tcp_options field, grouped and counted.
masscan_df = df[df['masscan'] == True]
masscan_grouped = masscan_df.groupby('tcp_options').size().reset_index(name='count').sort_values(by='count', ascending=False)
print("Unique TCP options in Masscan traffic:")
print(masscan_grouped)

In [ ]:
# This cell shows Unicorn packets, only the tcp_options field, grouped and counted.
unicorn_df = df[df['unicorn'] == True]
unicorn_grouped = unicorn_df.groupby('tcp_options').size().reset_index(name='count').sort_values(by='count', ascending=False)
print("Unique TCP options in Unicorn traffic:")
print(unicorn_grouped)

In [ ]:
# This cell shows packets that have not been classified in any category, only the tcp_options field, grouped and counted.
no_class_df = df[~df[['mirai', 'zmap', 'masscan', 'hajime', 'unicorn']].any(axis=1)]
no_class_grouped = no_class_df.groupby('tcp_options').size().reset_index(name='count').sort_values(by='count', ascending=False)
print("Unique TCP options in UNCLASSIFIED traffic:")
print(no_class_grouped)

In [ ]:
# Records that have tcp_options "020405b40103030801010402" or "020405a00103030801010402"
df_tcpoptions = df[(df['tcp_options'] == "020405b40103030801010402") | (df['tcp_options'] == "020405a00103030801010402")]
df_tcpoptions

In [ ]:
# Packets from df_tcpoptions that have the same source, destination and port, aggregated and counted
grouped_tcpoptions = df_tcpoptions.groupby(['ip_src', 'ip_dst', 'tcp_dport']).size().reset_index(name='count').sort_values(by='count', ascending=False) 
print("Grouping by ip_src, ip_dst, tcp_dport for specific TCP options:")
print(grouped_tcpoptions)

In [ ]:
grouped_tcpoptions[grouped_tcpoptions['count'] == 1]

In [ ]:
# Rows that are Hajime and show the most common destination ports in sorted order
hajime_df = df[df['tcp_window'] == '14600']

# Group by destination port and count occurrences
hajime_port_counts = hajime_df[['tcp_dport', 'ip_src']].value_counts().reset_index()
hajime_port_counts.columns = ['tcp_dport', 'ip_src', 'count']
hajime_port_counts.head(40)

In [ ]:
# The following cell gets all records that are probable Hajime
# Source IP addresses with more than 200 connections to common Hajime ports are taken
# Then those IP addresses are searched in the original DataFrame for any destination port

hajime_common_ports = [22, 23, 2222, 5555, 7547, 8291]

# Filter only rows with common Hajime ports
hajime_filtered = hajime_port_counts[hajime_port_counts['tcp_dport'].isin(hajime_common_ports)]

# Group by ip_src and sum the counts
hajime_ips_summary = hajime_filtered.groupby('ip_src')['count'].sum().reset_index()
hajime_ips_summary.columns = ['ip_src', 'total_count']

# Get IPs that have more than 200 total connections to common ports
hajime_common_ips = hajime_ips_summary[hajime_ips_summary['total_count'] > 20]['ip_src'].unique()

print(f"Hajime IPs found: {len(hajime_common_ips)}")
print(f"IPs: {hajime_common_ips}")

# Get all records from those IPs in the original DataFrame
hajime_full_df = df[df['ip_src'].isin(hajime_common_ips)]
hajime_full_df

In [ ]:
hajime_common_ips

In [ ]:
# I want to see rows that are not Hajime, nor possible Hajime, nor Mirai, nor ZMap, nor Masscan
df_no_conocidas = df[(df['hajime'] != True) & (df['hajime_possible'] != True) & (df['mirai'] != True) & (df['masscan'] != True)]
df_no_conocidas

```
6	2025-10-21 00:00:00.237600088	4	0x00			0	0	52	22492	49	...	020405b40103030801010402		1761004800.237600000_6	{'ip_ihl_ok': True, 'ip_orig_flags_ok': True, ...	False	False	2604032248	False	False	False
10	2025-10-21 00:00:00.344327927	4	0x00			0	0	52	26281	49	...	020405b40103030801010402		1761004800.344328000_10	{'ip_ihl_ok': True, 'ip_orig_flags_ok': True, ...	False	False	2604032175	False	False	False
11	2025-10-21 00:00:00.397784948	4	0x00			0	0	52	29971	48	...	020405b40103030801010402		1761004800.397785000_11	{'ip_ihl_ok': True, 'ip_orig_flags_ok': True, ...	False	False	2604032102	False	False	False
14	2025-10-21 00:00:00.417854071	4	0x00			0	0	52	51655	49	...	020405b40103030801010402		1761004800.417854000_14	{'ip_ihl_ok': True, 'ip_orig_flags_ok': True, ...	False	False	2604032007	False	False	False
18	2025-10-21 00:00:00.569519043	4	0x00			0	0	52	29002	49	...	020405b40103030801010402		1761004800.569519000_19	{'ip_ihl_ok': True, 'ip_orig_flags_ok': True, ...	False	False	2604032240	False	False	False
...	...	...	...	...	...	...	...	...	...	...	...	...	...	...	...	...	...	...	...	...	...
1670333	2025-10-21 23:59:59.527405977	4	0x00			0	0	60	31946	49	...	020405b4080a4445414400000000030301040200		1761091199.527406000_1793386	{'ip_ihl_ok': True, 'ip_orig_flags_ok': True, ...	False	False	2604032135	False	False	False
1670337	2025-10-21 23:59:59.655347109	4	0xa4			41	0	52	48478	49	...	0204059c0103030801010402		1761091199.655347000_1793394	{'ip_ihl_ok': True, 'ip_orig_flags_ok': True, ...	False	False	2604032198	False	False	False
1670338	2025-10-21 23:59:59.704593897	4	0x00			0	0	60	2272	51	...	020405b40402080a68f688fc000000000103030a		1761091199.704594000_1793395	{'ip_ihl_ok': True, 'ip_orig_flags_ok': True, ...	False	False	2604032189	False	False	False
1670339	2025-10-21 23:59:59.819652081	4	0xa4			41	0	60	57677	48	...	020405b40402080a68f68907000000000103030a		1761091199.819652000_1793396	{'ip_ihl_ok': True, 'ip_orig_flags_ok': True, ...	False	False	2604032234	False	False	False
1670340	2025-10-21 23:59:59.841208935	4	0x00			0	0	52	12552	47	...	020405b40103030801010402		1761091199.841209000_1793397
```

In [ ]:
len(df_no_conocidas)

In [ ]:
# From the unknown ones, I want to group tcp_options and see which are the most common
df_no_conocidas['tcp_options'] = df_no_conocidas['tcp_options'].fillna('')
options_counts = df_no_conocidas['tcp_options'].value_counts().reset_index()
options_counts.columns = ['tcp_options', 'count']
options_counts_sorted = options_counts.sort_values('count', ascending=False)
options_counts_sorted.head(20)

In [ ]:
df_total = df.copy()
df_total['tcp_options'] = df_total['tcp_options'].fillna('')
options_counts_total = df_total['tcp_options'].value_counts().reset_index()
options_counts_total.columns = ['tcp_options', 'count']
options_counts_sorted_total = options_counts_total.sort_values('count', ascending=False)
options_counts_sorted_total.head(20)

In [ ]:
# I want to compare the top 20 tcp_options from the unknown ones with the total dataframe and see what the differences are
options_counts_sorted_20 = options_counts_sorted ## .head(20)

# Merge with ALL counts from total, not just top 20, so it finds all
comparison_df = options_counts_sorted_20.merge(options_counts_sorted_total, on='tcp_options', how='left', suffixes=('_unknown', '_total')).fillna(0)
comparison_df['classified'] = comparison_df['count_total'] - comparison_df['count_unknown']
comparison_df = comparison_df.sort_values('classified', ascending=False)
comparison_df

In [ ]:
comparison_df = comparison_df.sort_values('count_unknown', ascending=False)
comparison_df

In [ ]:
# Filter df with tpc_options with value 020405b40103030801010402
df_options = df[df['tcp_options'] == '020405b40103030801010402']
df_options[['ip_src', 'ip_dst', 'tcp_dport']]

In [ ]:
# Group df_options by ip_src and tcp_dport
df_options_grouped = df_options.groupby(['ip_src', 'tcp_dport']).size().reset_index(name='count').sort_values(by='count', ascending=False)
df_options_grouped

In [ ]:
# Filter rows where count == 1 and group by ip_src with list of tcp_dport
df_options_single = df_options_grouped[df_options_grouped['count'] == 1]
df_options_single_grouped = df_options_single.groupby('ip_src')['tcp_dport'].agg(list).reset_index()
df_options_single_grouped.columns = ['ip_src', 'tcp_dport_list']
# Sort each list individually
df_options_single_grouped['tcp_dport_list'] = df_options_single_grouped['tcp_dport_list'].apply(sorted)
df_options_single_grouped['port_count'] = df_options_single_grouped['tcp_dport_list'].apply(len)
df_options_single_grouped = df_options_single_grouped.sort_values(by='port_count', ascending=False)
df_options_single_grouped